In [1]:
import os
from dotenv import load_dotenv
load_dotenv('env')

True

### User Baseline Embedding

In [11]:
from pinecone import ServerlessSpec, Pinecone

pc = Pinecone(api_key=os.environ.get("PINECONE_KEYS1"))
user_index_name = "users-index"

if user_index_name not in [x['name'] for x in pc.list_indexes().to_dict()['indexes']]:
    pc.create_index(
        name=user_index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud='aws', 
            region='us-east-1'
        ) 
    ) 

while not pc.describe_index(user_index_name).status['ready']:
    time.sleep(1)

user_index = pc.Index(user_index_name)

In [15]:
N = 10000
item_index = pc.Index('quotes-index')

In [17]:
all_vecs = item_index.query(
        vector= [0] * 384,
        top_k=N,
        include_values=True,
        include_metadata=False
    )

In [20]:
import numpy as np
average_emb = np.array([0.]*384)
for match in all_vecs['matches']:
    average_emb += np.array(match['values'])
average_emb = average_emb/N

In [36]:
user_index.upsert(vectors=[{
    "id": 'user_0',
    "values": average_emb.tolist(),
}])

{'upserted_count': 1}

### Pretraining Ranker (& 2nd Filter)

In [ ]:
import torch
import pinecone
import numpy as np
from collections import defaultdict

# Initialize Pinecone for vector storage
pinecone.init(api_key="YOUR_PINECONE_API_KEY", environment="us-west1-gcp")
index = pinecone.Index("quotes")

# User Profile (Dynamic Updates)
user_profile = {
    "embedding": torch.zeros(768),  # Assume 768-dim embeddings (e.g., BERT-based)
    "author_weights": defaultdict(float),
    "keyword_weights": defaultdict(float),
    "complexity_preference": 0.0,
    "length_preference": 0.0,
    "engagement_count": 0,
}

def update_user_profile(quote, liked=False, time_spent=0):
    """
    Update user profile based on engagement signals.
    - Liked quotes get higher weight.
    - Time spent reading (normalized by length) also contributes.
    """
    global user_profile
    quote_embedding = torch.tensor(quote["embedding"])
    quote_length = quote["length"]
    author = quote["author"]
    keywords = quote["keywords"]
    complexity = quote["complexity"]

    # Engagement score: More weight for likes, but also consider time spent
    engagement_score = 1.0 if liked else min(0.5, time_spent / quote_length)
    
    # Update user embedding (moving average approach)
    user_profile["embedding"] = (
        user_profile["embedding"] * user_profile["engagement_count"] + engagement_score * quote_embedding
    ) / (user_profile["engagement_count"] + 1)
    
    # Update author preference
    user_profile["author_weights"][author] += engagement_score
    
    # Update keyword preference
    for keyword in keywords:
        user_profile["keyword_weights"][keyword] += engagement_score
    
    # Update complexity & length preference
    user_profile["complexity_preference"] = (
        user_profile["complexity_preference"] * user_profile["engagement_count"] + engagement_score * complexity
    ) / (user_profile["engagement_count"] + 1)
    
    user_profile["length_preference"] = (
        user_profile["length_preference"] * user_profile["engagement_count"] + engagement_score * quote_length
    ) / (user_profile["engagement_count"] + 1)
    
    user_profile["engagement_count"] += 1


def recommend_quotes(n=5):
    """
    Retrieve top N recommended quotes based on dynamic scoring.
    """
    user_vector = user_profile["embedding"].numpy().tolist()
    results = index.query(user_vector, top_k=50, include_metadata=True)
    
    scored_quotes = []
    for match in results["matches"]:
        quote = match["metadata"]
        quote_embedding = torch.tensor(match["vector"])
        author = quote["author"]
        keywords = quote["keywords"]
        complexity = quote["complexity"]
        length = quote["length"]
        
        # Compute cosine similarity
        similarity = torch.nn.functional.cosine_similarity(
            torch.tensor(user_vector), quote_embedding, dim=0
        ).item()
        
        # Compute weighted relevance score
        author_weight = user_profile["author_weights"].get(author, 0)
        keyword_weight = sum(user_profile["keyword_weights"].get(k, 0) for k in keywords)
        complexity_score = -abs(user_profile["complexity_preference"] - complexity)
        length_score = -abs(user_profile["length_preference"] - length)
        
        score = (
            0.5 * similarity + 0.2 * author_weight + 0.2 * keyword_weight +
            0.05 * complexity_score + 0.05 * length_score
        )
        scored_quotes.append((quote, score))
    
    # Sort by score and return top N
    scored_quotes.sort(key=lambda x: x[1], reverse=True)
    return [q[0] for q in scored_quotes[:n]]

# Example Usage
# quote_example = {
#     "embedding": np.random.rand(768).tolist(),
#     "author": "Albert Einstein",
#     "keywords": ["knowledge", "science"],
#     "complexity": 8.5,
#     "length": 120
# }
# update_user_profile(quote_example, liked=True, time_spent=15)
# recommendations = recommend_quotes(n=5)
